# HotpotQA — Dataset Profile

This notebook describes the **HotpotQA fullwiki validation population**
used in the Context Matters project.

The goal is to understand the dataset itself:

- what kind of questions HotpotQA contains;
- why it is a multi-hop QA benchmark;
- bridge versus comparison questions;
- supporting-fact annotations;
- question difficulty;
- answer characteristics;
- per-question context structure;
- why HotpotQA is useful for evaluating retrieval-augmented generation.

This notebook does **not** report Sprint-1, Sprint-2, or Sprint-3 model results.

## 1. Frozen Dataset Source

For answer correctness, the project uses:

- Dataset: `hotpotqa/hotpot_qa`
- Configuration: `fullwiki`
- Split: `validation`
- Frozen revision: `1908d6afbbead072334abe2965f91bd2709910ab`
- Questions: **7,405**

These IDs are aligned exactly with the project's frozen official BEIR
HotpotQA test-query manifest.

In [ ]:
from pathlib import Path
import sys
import json

import pandas as pd

REPO = Path.cwd().resolve()

while REPO != REPO.parent and not (REPO / "artifacts").is_dir():
    REPO = REPO.parent

assert (REPO / "artifacts").is_dir(), "Could not locate repository root"

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("Repository:", REPO)

In [ ]:
from datasets import load_dataset, disable_progress_bar

disable_progress_bar()

GOLD_REVISION = "1908d6afbbead072334abe2965f91bd2709910ab"

MANIFEST_PATH = (
    REPO
    / "artifacts"
    / "sample_manifests"
    / "hotpotqa_official_test_full_manifest.json"
)

manifest_artifact = json.loads(
    MANIFEST_PATH.read_text(encoding="utf-8")
)

manifest_entries = (
    manifest_artifact["scientific_payload"]["entries"]
)

manifest_ids = [
    str(entry["source_sample_id"])
    for entry in manifest_entries
]

dataset = load_dataset(
    "hotpotqa/hotpot_qa",
    "fullwiki",
    split="validation",
    revision=GOLD_REVISION,
    download_mode="reuse_dataset_if_exists",
)

rows = tuple(dataset)

dataset_ids = [str(row["id"]) for row in rows]

assert len(rows) == 7_405
assert len(manifest_ids) == 7_405
assert len(set(manifest_ids)) == 7_405
assert set(dataset_ids) == set(manifest_ids)

print("HotpotQA validation: PASS")
print(f"Questions:       {len(rows):,}")
print("Configuration:   fullwiki")
print("Split:           validation")
print("Frozen revision:", GOLD_REVISION)
print("Manifest IDs aligned exactly: YES")
print("LLM/API calls made: 0")

## 2. Dataset Schema

Each HotpotQA example contains:

| Field | Meaning |
|---|---|
| `id` | Unique question identifier |
| `question` | Natural-language question |
| `answer` | Gold answer |
| `type` | `bridge` or `comparison` |
| `level` | Difficulty label |
| `supporting_facts` | Gold supporting document titles and sentence IDs |
| `context` | Fullwiki candidate documents and their sentences |

The supporting-fact annotations are especially important because HotpotQA
was designed to require evidence from multiple pieces of information.

In [ ]:
schema = pd.DataFrame(
    [
        ["id", "string", "Unique HotpotQA question ID"],
        ["question", "string", "Natural-language question"],
        ["answer", "string", "Gold answer"],
        ["type", "string", "bridge or comparison"],
        ["level", "string", "Difficulty label"],
        [
            "supporting_facts.title",
            "list[string]",
            "Titles containing gold supporting facts",
        ],
        [
            "supporting_facts.sent_id",
            "list[int]",
            "Sentence indexes of gold supporting facts",
        ],
        [
            "context.title",
            "list[string]",
            "Titles in the provided fullwiki context",
        ],
        [
            "context.sentences",
            "list[list[string]]",
            "Sentences for each provided context document",
        ],
    ],
    columns=["Field", "Type", "Description"],
)

print(schema.to_string(index=False))

## 3. What Makes HotpotQA Multi-Hop?

HotpotQA is designed so that many questions cannot be answered from a single
isolated fact.

A system often has to connect evidence across multiple entities or documents.

Two major question types are provided:

- **Bridge:** one fact identifies an intermediate entity, which is then used
  to obtain another fact.
- **Comparison:** information about two entities must be gathered and compared.

This makes HotpotQA useful for studying whether retrieved context contains
the right combination of evidence for reasoning.

## 4. Example Question

In [ ]:
example = rows[0]

print("Question ID:")
print(example["id"])

print("\nQuestion:")
print(example["question"])

print("\nGold answer:")
print(example["answer"])

print("\nQuestion type:")
print(example["type"])

print("\nDifficulty:")
print(example["level"])

print("\nSupporting facts:")
for title, sent_id in zip(
    example["supporting_facts"]["title"],
    example["supporting_facts"]["sent_id"],
):
    print(f"- {title} | sentence {sent_id}")

## 5. Question-Type Distribution

In [ ]:
type_counts = (
    pd.Series(
        [row["type"] for row in rows],
        name="Question type",
    )
    .value_counts()
    .rename_axis("Question type")
    .reset_index(name="Questions")
)

type_counts["Percent"] = (
    100 * type_counts["Questions"] / len(rows)
)

print(type_counts.to_string(index=False))

## 6. Difficulty Distribution

The frozen evaluation population used by this project consists entirely of
questions labeled **hard**.

In [ ]:
level_counts = (
    pd.Series(
        [row["level"] for row in rows],
        name="Difficulty",
    )
    .value_counts()
    .rename_axis("Difficulty")
    .reset_index(name="Questions")
)

level_counts["Percent"] = (
    100 * level_counts["Questions"] / len(rows)
)

print(level_counts.to_string(index=False))

## 7. Supporting Facts

Each question contains explicit supporting-fact annotations.

A supporting fact is represented by:

- a Wikipedia document title;
- a sentence index within that document.

These annotations describe the evidence intended to support the gold answer.

In [ ]:
supporting_fact_counts = pd.Series(
    [
        len(row["supporting_facts"]["title"])
        for row in rows
    ],
    name="Supporting facts",
)

print(
    supporting_fact_counts
    .describe()
    .round(2)
    .to_string()
)

print(
    "\nTotal supporting-fact annotations:",
    f"{supporting_fact_counts.sum():,}",
)

## 8. Number of Distinct Supporting Documents

Multiple supporting facts can come from the same document, so the number of
supporting facts and the number of distinct supporting documents are not
always identical.

In [ ]:
supporting_doc_counts = pd.Series(
    [
        len(set(row["supporting_facts"]["title"]))
        for row in rows
    ],
    name="Distinct supporting documents",
)

print(
    supporting_doc_counts
    .describe()
    .round(2)
    .to_string()
)

print("\nDistribution:")
print(
    supporting_doc_counts
    .value_counts()
    .sort_index()
    .rename_axis("Documents")
    .to_string()
)

## 9. Provided Fullwiki Context Structure

The original HotpotQA `fullwiki` data also provides a set of candidate
Wikipedia documents for each question.

This field helps describe the benchmark's original data structure.

**Important:** this per-question `context` field is not the retrieval corpus
used by our RAG retrievers.

In [ ]:
context_doc_counts = pd.Series(
    [
        len(row["context"]["title"])
        for row in rows
    ],
    name="Provided context documents",
)

print(
    context_doc_counts
    .describe()
    .round(2)
    .to_string()
)

zero_context = int((context_doc_counts == 0).sum())

print(
    "\nQuestions with zero provided context documents:",
    zero_context,
)

## 10. Provided Context vs Project Retrieval Corpus

Two different objects must not be confused.

### Original HotpotQA `context`

This is the per-question context field stored in the `fullwiki` dataset.

### Context Matters retrieval corpus

Our retrieval systems search the complete frozen **BEIR HotpotQA corpus**:

**5,233,329 documents**

BM25, DPR, Contriever, and ColBERTv2 retrieve from that shared corpus.

Therefore the original per-question context is useful for understanding
HotpotQA, but it is not used as a small substitute corpus for the project's
retrieval experiment.

## 11. Answer Characteristics

HotpotQA answers can be:

- entity names;
- dates;
- places;
- people;
- short factual phrases;
- `yes` or `no`.

The benchmark therefore evaluates concise factual answer generation rather
than long-form explanation generation.

In [ ]:
answers = pd.Series(
    [str(row["answer"]).strip() for row in rows],
    name="Answer",
)

answer_words = answers.str.split().str.len()

answer_summary = pd.DataFrame(
    {
        "Metric": [
            "Questions",
            "Unique answer strings",
            "Mean answer words",
            "Median answer words",
            "Maximum answer words",
            "yes answers",
            "no answers",
        ],
        "Value": [
            len(answers),
            answers.nunique(),
            round(answer_words.mean(), 2),
            float(answer_words.median()),
            int(answer_words.max()),
            int((answers.str.lower() == "yes").sum()),
            int((answers.str.lower() == "no").sum()),
        ],
    }
)

print(answer_summary.to_string(index=False))

## 12. Question Length

In [ ]:
question_words = pd.Series(
    [
        len(row["question"].split())
        for row in rows
    ],
    name="Question words",
)

print(
    question_words
    .describe()
    .round(2)
    .to_string()
)

## 13. Example Bridge Question

In [ ]:
bridge = next(
    row for row in rows
    if row["type"] == "bridge"
)

print("Question:")
print(bridge["question"])

print("\nAnswer:")
print(bridge["answer"])

print("\nSupporting facts:")
for title, sent_id in zip(
    bridge["supporting_facts"]["title"],
    bridge["supporting_facts"]["sent_id"],
):
    print(f"- {title} | sentence {sent_id}")

## 14. Example Comparison Question

In [ ]:
comparison = next(
    row for row in rows
    if row["type"] == "comparison"
)

print("Question:")
print(comparison["question"])

print("\nAnswer:")
print(comparison["answer"])

print("\nSupporting facts:")
for title, sent_id in zip(
    comparison["supporting_facts"]["title"],
    comparison["supporting_facts"]["sent_id"],
):
    print(f"- {title} | sentence {sent_id}")

## 15. Why HotpotQA Is Useful for This Project

HotpotQA is valuable for the Context Matters research question because
multi-hop reasoning depends not only on retrieving relevant documents, but
on retrieving the **right combination of evidence**.

This gives diversification a meaningful role to test.

A diversified context could:

- expose complementary pieces of evidence;
- reduce redundant passages;
- improve multi-hop reasoning.

But it could also:

- replace a highly relevant passage with a less relevant one;
- introduce distracting entities;
- increase unsupported reasoning or hallucination.

HotpotQA therefore provides a strong setting for testing the trade-off
between relevance and diversity.

## 16. Important Dataset Limitations

Important limitations include:

- HotpotQA is constructed around Wikipedia evidence;
- supporting-fact annotations represent benchmark evidence rather than every
  possible valid reasoning path;
- concise answers do not measure explanation quality by themselves;
- the frozen project population contains only `hard` questions;
- multi-hop success depends jointly on retrieval and generation;
- Wikipedia-based QA does not represent all real-world RAG applications.

## 17. Dataset Summary

In [ ]:
summary = pd.DataFrame(
    [
        ["Dataset", "HotpotQA"],
        ["Configuration", "fullwiki"],
        ["Split", "validation"],
        ["Questions", f"{len(rows):,}"],
        ["Bridge questions", f"{sum(r['type'] == 'bridge' for r in rows):,}"],
        [
            "Comparison questions",
            f"{sum(r['type'] == 'comparison' for r in rows):,}",
        ],
        ["Difficulty", "hard"],
        ["Gold supporting facts", "Yes"],
        ["Gold supporting document titles", "Yes"],
        ["Project retrieval corpus", "5,233,329 BEIR documents"],
        ["Role in project", "Multi-hop reasoning dataset"],
    ],
    columns=["Property", "Value"],
)

print(summary.to_string(index=False))

## 18. Reproducibility Check

In [ ]:
checks = pd.DataFrame(
    [
        [
            "Dataset contains exactly 7,405 questions",
            len(rows) == 7_405,
        ],
        [
            "Manifest contains exactly 7,405 IDs",
            len(manifest_ids) == 7_405,
        ],
        [
            "Dataset and manifest ID sets match exactly",
            set(dataset_ids) == set(manifest_ids),
        ],
        [
            "All IDs are unique",
            len(set(dataset_ids)) == 7_405,
        ],
        [
            "Question types are bridge/comparison only",
            set(row["type"] for row in rows)
            == {"bridge", "comparison"},
        ],
        [
            "All frozen questions are hard",
            set(row["level"] for row in rows)
            == {"hard"},
        ],
        [
            "All questions are non-empty",
            all(bool(row["question"].strip()) for row in rows),
        ],
        [
            "All gold answers are non-empty",
            all(bool(str(row["answer"]).strip()) for row in rows),
        ],
    ],
    columns=["Check", "PASS"],
)

assert checks["PASS"].all(), checks[~checks["PASS"]]

print(checks.to_string(index=False))

print("\nPASS: HotpotQA dataset profile is reproducible")
print("LLM/API calls made by this notebook: 0")

## Reproducibility

This notebook uses the same frozen HotpotQA question population and
gold-answer source as the main project.

It performs descriptive dataset analysis only and makes no LLM/API calls.